In [ ]:
import carla
from carla import Transform, Location, Rotation 
import random
import sys

sys.path.append("./CARLA_0.9.16") # Add the folder to the path

from PythonAPI.carla.agents.navigation.basic_agent import BasicAgent
from PythonAPI.carla.agents.navigation.behavior_agent import BehaviorAgent


client = carla.Client('localhost', 2000)
client.set_timeout(5.0)
world = client.get_world()
settings = world.get_settings()
settings.fixed_delta_seconds = 0.05 # Update the world every 0.05 seconds 
#^ fps: 20 
#for calculations and rendering we will do two things 
#we will put the simulation in synchronous mode 
#Synchronous mode means that the simulation will 
# wait for the client to send a tick before it updates the world.

#in this simulation we need to make sure 
#that the server waits on the calculations to be performed before it updates the world 

#The physics of the world will have to be more precise than 0.05 seconds though 
#we will split the 0.05 seconds into smaller steps 
#we will use substepping of 0.01 seconds and a maximum of 10 substeps per tick


max_substep_delta_time = 0.01
max_substeps = 10
settings.synchronous_mode = True # Enables 
#we need to make sure that the physics engine has simulated enough steps to cover the fixed delta time of 0.05 seconds

world.apply_settings(settings)

dt = settings.fixed_delta_seconds

In [ ]:
transform = Transform(Location(x=50, y=27, z=1), Rotation(yaw=180))
spawn_points = world.get_map().get_spawn_points()

bp_lib = world.get_blueprint_library()

# Option A: exact id (b
bp = bp_lib.find('vehicle.micro.microlino')

actor = world.try_spawn_actor(bp, transform)
print("spawned:", actor)

if actor is None:
    raise RuntimeError("Vehicle failed to spawn")

agent = BehaviorAgent(actor, behavior='aggressive')   # or BasicAgent(actor)

t = actor.get_transform()
behind = t.location - t.get_forward_vector() * 8 + carla.Location(z=3)


spectator = world.get_spectator()
spectator.set_transform(carla.Transform(
    behind,
    carla.Rotation(pitch=-10, yaw=t.rotation.yaw)
))

In [ ]:
imu_bp = bp_lib.find('sensor.other.imu')

imu_transform = carla.Transform(
    carla.Location(x=50.0, y=27.0, z=1.5)  # on top of vehicle
)

imu = world.spawn_actor(imu_bp, imu_transform, attach_to=actor)

gps_bp = bp_lib.find('sensor.other.gnss')

gps_transform = carla.Transform(
    carla.Location(x=50.0, y=27.0, z=1.5)
)

gps = world.spawn_actor(gps_bp, gps_transform, attach_to=actor)

In [ ]:
def imu_callback(data):
    # print("Accel:", data.accelerometer)
    # print("Gyro:", data.gyroscope)
    global imu_data
    imu_data = data



def gps_callback(data):
    # print("Lat:", data.latitude, "Lon:", data.longitude)
    global gps_data
    gps_data = data
imu.listen(imu_callback)
gps.listen(gps_callback)

In [ ]:
from PythonAPI.carla.agents.navigation.global_route_planner import GlobalRoutePlanner

destination = random.choice(spawn_points).location
agent.set_destination(destination)
m = world.get_map()
debug = world.debug
origin = actor.get_location()
grp = GlobalRoutePlanner(m, sampling_resolution=2.0)
route = grp.trace_route(origin, destination) #FIXME: end_location is not defined, should be destination 
# 1) Draw sparse GRP graph nodes
for current_waypoint, road_option in route:
    world.debug.draw_point(current_waypoint.transform.location + carla.Location(z=0.5), life_time = 50 )

In [ ]:
vx, vy, calculated_x, calculated_y, calculated_yaw = 0, 0, origin.x, origin.y, 0
while True:
    world.tick() # Wait for the server to tick before we do anything
    if imu_data is None:
        continue
    if gps_data is None:
        continue
    if agent.done():
        print("The target has been reached, stopping the simulation")
        break
    
    current_imu = imu_data 
    current_gps = gps_data
    
    acceleration = current_imu.accelerometer
    angular_velocity = current_imu.gyroscope
    location = current_gps.transform.location
    gps_x = location.x
    gps_y = location.y
    gps_z = location.z
    
    #TODO: recursive for now: kalman later 
    
    vx = vx + acceleration.x * dt
    vy = vy + acceleration.y * dt
    calculated_x = calculated_x + vx * dt
    calculated_y = calculated_y + vy * dt
    calculated_yaw = calculated_yaw + angular_velocity.z * dt
    
    alpha = 0.5 # weight for GPS vs IMU
    
    x = alpha * gps_x + (1 - alpha) * calculated_x
    y = alpha * gps_y + (1 - alpha) * calculated_y
    
    control = vehicle_pid.run_step(target_speed, waypoint) #FIXME: define target_speed and waypoint and call vechile_pid
    actor.apply_control(control)

    t = actor.get_transform()
    # behind + a little up
    behind = t.location - t.get_forward_vector() * 8 + carla.Location(z=3)

    spectator.set_transform(carla.Transform(
        behind,
        carla.Rotation(pitch=-10, yaw=t.rotation.yaw)
    ))
    
    

In [ ]:
actor.destroy()
imu.destroy()
gps.destroy()